# 7-40: фреймворк линии, превью и символических сборных

Ноутбук сделан как заготовка: запускается на мок-данных, но все входы вынесены в явные файлы. Когда появятся Telegram-логи и примеры превью, достаточно заменить входные файлы в блоке параметров.

## Входы и выходы

- `data/football/telegram_7_40_export.json` - экспорт Telegram Desktop в JSON.
- `data/football/telegram_7_40_messages.csv` - альтернативный нормализованный файл с колонками `date`, `author`, `text`.
- `data/football/coach_previews.csv` - превью тренеров: `round_id`, `match`, `coach`, `team`, `text`.
- `data/football/player_round_stats.csv` - статистика игроков для символических сборных.
- `output/football/7_40_selected_line.xlsx` - выбранная линия из 7 вариантов.
- `output/football/7_40_symbolic_squad.xlsx` - кандидаты и итоговая символическая сборная.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd

PROJECT_ROOT = Path('/Users/aozolotarev/Documents/personal')
DATA_DIR = PROJECT_ROOT / 'data' / 'football'
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'football'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARAMS = {
    'line_size': 7,
    'max_per_category': 3,
    'history_half_life_rounds': 12,
    'weights': {
        'history_hit_rate': 0.45,
        'preview_signal': 0.30,
        'manual_prior': 0.25,
    },
    'symbolic_squad_size': 5,
}

DATA_DIR.mkdir(parents=True, exist_ok=True)
PARAMS

In [ ]:
VARIANTS = [
    (1, 'Команда 1 выиграет матч', 'result'),
    (2, 'Команда 2 выиграет матч', 'result'),
    (3, 'Матч закончится вничью', 'result'),
    (4, 'Команда 1 выиграет первый тайм', 'half_result'),
    (5, 'Команда 2 выиграет первый тайм', 'half_result'),
    (6, 'Первый тайм закончится вничью', 'half_result'),
    (7, 'Команда 1 выиграет второй тайм', 'half_result'),
    (8, 'Команда 2 выиграет второй тайм', 'half_result'),
    (9, 'Второй тайм закончится вничью', 'half_result'),
    (10, 'Первый тайм будет результативнее второго', 'goals_by_half'),
    (11, 'Второй тайм будет результативнее первого', 'goals_by_half'),
    (12, 'Таймы будут равны по результативности', 'goals_by_half'),
    (13, 'В матче не будет забито голов', 'goals_total'),
    (14, 'В матче будет забито 1 или 2 гола', 'goals_total'),
    (15, 'В матче будет забито 3 или 4 гола', 'goals_total'),
    (16, 'В матче будет забито больше 4 голов', 'goals_total'),
    (17, 'Команда 1 не пропустит', 'clean_sheet'),
    (18, 'Команда 2 не пропустит', 'clean_sheet'),
    (19, 'Команда 1 будет проигрывать, но не проиграет матч', 'comeback'),
    (20, 'Команда 2 будет проигрывать, но не проиграет матч', 'comeback'),
    (21, 'Обе команды забьют в первом тайме', 'both_score'),
    (22, 'Обе команды забьют во втором тайме', 'both_score'),
    (23, 'Первый мяч будет забит с 1 по 20 минуту', 'first_goal'),
    (24, 'Первый мяч будет забит с 21 по 45+ минуту', 'first_goal'),
    (25, 'Первый мяч будет забит во втором тайме', 'first_goal'),
    (26, 'Победный мяч будет забит в первом тайме', 'winning_goal'),
    (27, 'Победный мяч будет забит с 46 по 70 минуту', 'winning_goal'),
    (28, 'Победный мяч будет забит с 71 по 90+ минуту', 'winning_goal'),
    (29, 'Разница голов составит 1 мяч', 'goal_difference'),
    (30, 'Разница голов составит 2 мяча', 'goal_difference'),
    (31, 'Разница голов составит 3 и больше мячей', 'goal_difference'),
    (32, 'Игрок любой команды забьет 2 и больше мячей', 'player_goals'),
    (33, 'В матче будет 3 или меньше предупреждений', 'discipline'),
    (34, 'В матче будет 4-5 предупреждений', 'discipline'),
    (35, 'В матче будет 6 или больше предупреждений', 'discipline'),
    (36, 'В матче будет хотя бы одно удаление или автогол', 'discipline'),
    (37, 'В матче будет назначен хотя бы один пенальти', 'discipline'),
    (38, 'В матче будет 5 или меньше замен', 'substitutions'),
    (39, 'Хотя бы одна замена произойдет в первом тайме или перерыве', 'substitutions'),
    (40, 'Вышедшие на замену игроки забьют хотя бы один гол', 'substitutions'),
]

variants = pd.DataFrame(VARIANTS, columns=['variant_id', 'variant_text', 'category'])
variants.head(10)

In [ ]:
def read_telegram_export(path: Path) -> pd.DataFrame:
    """Read Telegram Desktop JSON export into date/author/text rows."""
    with path.open('r', encoding='utf-8') as f:
        raw = json.load(f)

    rows = []
    for msg in raw.get('messages', []):
        text = msg.get('text', '')
        if isinstance(text, list):
            text = ''.join(part if isinstance(part, str) else part.get('text', '') for part in text)
        rows.append({
            'date': msg.get('date'),
            'author': msg.get('from'),
            'text': text,
        })
    return pd.DataFrame(rows)


def load_telegram_messages() -> pd.DataFrame:
    json_path = DATA_DIR / 'telegram_7_40_export.json'
    csv_path = DATA_DIR / 'telegram_7_40_messages.csv'
    if json_path.exists():
        return read_telegram_export(json_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    return pd.DataFrame([
        {'date': '2026-04-01', 'author': 'moderator', 'text': 'Итоги: зашли варианты 1, 14, 24, 29, 34'},
        {'date': '2026-04-08', 'author': 'moderator', 'text': 'Зашло: 3 15 22 27 33 37'},
    ])


VARIANT_PATTERN = re.compile(r'(?<!\d)([1-9]|[1-3]\d|40)(?!\d)')


def extract_variant_ids(text: str) -> list[int]:
    ids = [int(x) for x in VARIANT_PATTERN.findall(str(text))]
    return sorted(set(x for x in ids if 1 <= x <= 40))


messages = load_telegram_messages()
messages['variant_ids'] = messages['text'].map(extract_variant_ids)
messages

In [ ]:
def build_history(messages: pd.DataFrame) -> pd.DataFrame:
    exploded = messages.explode('variant_ids').dropna(subset=['variant_ids']).copy()
    if exploded.empty:
        return variants.assign(history_hits=0, history_hit_rate=0.0)
    counts = exploded['variant_ids'].astype(int).value_counts().rename_axis('variant_id').reset_index(name='history_hits')
    out = variants.merge(counts, on='variant_id', how='left').fillna({'history_hits': 0})
    out['history_hit_rate'] = out['history_hits'] / max(out['history_hits'].sum(), 1)
    return out


def load_previews() -> pd.DataFrame:
    path = DATA_DIR / 'coach_previews.csv'
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame([
        {
            'round_id': 'demo-1',
            'match': 'Интер - Ливерпуль',
            'coach': 'demo',
            'team': 'demo team',
            'text': 'Ожидаю высокий темп, голы обеих команд и активную концовку.'
        }
    ])


SIGNALS = {
    'goals_total': ['гол', 'темп', 'атака', 'открытый', 'результатив'],
    'both_score': ['обе', 'забьют', 'обмен', 'моменты'],
    'first_goal': ['старт', 'первые минуты', 'давление', 'быстрый гол'],
    'winning_goal': ['концовка', 'после перерыва', 'замены', 'дожмут'],
    'discipline': ['карточ', 'жестк', 'фол', 'дерби', 'судья'],
    'substitutions': ['замен', 'скамейк', 'ротац'],
}


def score_preview_signals(previews: pd.DataFrame) -> pd.DataFrame:
    text = ' '.join(previews.get('text', pd.Series(dtype=str)).fillna('').astype(str)).lower()
    rows = []
    for category, keywords in SIGNALS.items():
        hits = sum(text.count(keyword) for keyword in keywords)
        rows.append({'category': category, 'preview_signal': min(hits / 5, 1.0)})
    return pd.DataFrame(rows)


history = build_history(messages)
previews = load_previews()
preview_signals = score_preview_signals(previews)
history.head(), preview_signals

In [ ]:
def score_variants(history: pd.DataFrame, preview_signals: pd.DataFrame, manual_prior: dict[int, float] | None = None) -> pd.DataFrame:
    manual_prior = manual_prior or {}
    w = PARAMS['weights']
    scored = history.merge(preview_signals, on='category', how='left').fillna({'preview_signal': 0.0})
    scored['manual_prior'] = scored['variant_id'].map(manual_prior).fillna(0.5)
    scored['score'] = (
        w['history_hit_rate'] * scored['history_hit_rate']
        + w['preview_signal'] * scored['preview_signal']
        + w['manual_prior'] * scored['manual_prior']
    )
    return scored.sort_values('score', ascending=False)


def select_line(scored: pd.DataFrame, n: int = 7, max_per_category: int = 3) -> pd.DataFrame:
    selected = []
    category_counts: dict[str, int] = {}
    for row in scored.to_dict('records'):
        category = row['category']
        if category_counts.get(category, 0) >= max_per_category:
            continue
        selected.append(row)
        category_counts[category] = category_counts.get(category, 0) + 1
        if len(selected) == n:
            break
    return pd.DataFrame(selected)


manual_prior = {14: 0.70, 15: 0.65, 22: 0.65, 29: 0.60, 34: 0.55}
scored_variants = score_variants(history, preview_signals, manual_prior=manual_prior)
selected_line = select_line(scored_variants, n=PARAMS['line_size'], max_per_category=PARAMS['max_per_category'])
selected_line[['variant_id', 'variant_text', 'category', 'score']]

In [ ]:
def load_player_round_stats() -> pd.DataFrame:
    path = DATA_DIR / 'player_round_stats.csv'
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame([
        {'round_id': 'demo-1', 'player': 'Игрок A', 'team': 'Team 1', 'variants_sent': 7, 'variants_hit': 4, 'rare_hits': 1, 'preview_bonus': 0.2, 'manual_bonus': 0.0},
        {'round_id': 'demo-1', 'player': 'Игрок B', 'team': 'Team 2', 'variants_sent': 7, 'variants_hit': 3, 'rare_hits': 2, 'preview_bonus': 0.4, 'manual_bonus': 0.0},
        {'round_id': 'demo-1', 'player': 'Игрок C', 'team': 'Team 3', 'variants_sent': 7, 'variants_hit': 5, 'rare_hits': 0, 'preview_bonus': 0.1, 'manual_bonus': 0.0},
        {'round_id': 'demo-1', 'player': 'Игрок D', 'team': 'Team 4', 'variants_sent': 7, 'variants_hit': 2, 'rare_hits': 1, 'preview_bonus': 0.7, 'manual_bonus': 0.0},
        {'round_id': 'demo-1', 'player': 'Игрок E', 'team': 'Team 5', 'variants_sent': 7, 'variants_hit': 4, 'rare_hits': 0, 'preview_bonus': 0.3, 'manual_bonus': 0.0},
        {'round_id': 'demo-1', 'player': 'Игрок F', 'team': 'Team 6', 'variants_sent': 7, 'variants_hit': 3, 'rare_hits': 1, 'preview_bonus': 0.2, 'manual_bonus': 0.0},
    ])


def rank_symbolic_squad(stats: pd.DataFrame, squad_size: int = 5) -> tuple[pd.DataFrame, pd.DataFrame]:
    required = {'player', 'team', 'variants_sent', 'variants_hit', 'rare_hits', 'preview_bonus', 'manual_bonus'}
    missing = required - set(stats.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    ranked = stats.copy()
    ranked['hit_rate'] = ranked['variants_hit'] / ranked['variants_sent'].clip(lower=1)
    ranked['symbolic_score'] = (
        10 * ranked['variants_hit']
        + 3 * ranked['rare_hits']
        + 2 * ranked['preview_bonus']
        + ranked['manual_bonus']
    )
    ranked = ranked.sort_values(['symbolic_score', 'hit_rate'], ascending=False)
    squad = ranked.head(squad_size).copy()
    squad['symbolic_slot'] = range(1, len(squad) + 1)
    return ranked, squad


player_stats = load_player_round_stats()
ranked_players, symbolic_squad = rank_symbolic_squad(player_stats, PARAMS['symbolic_squad_size'])
symbolic_squad

In [ ]:
line_path = OUTPUT_DIR / '7_40_selected_line.xlsx'
squad_path = OUTPUT_DIR / '7_40_symbolic_squad.xlsx'

with pd.ExcelWriter(line_path) as writer:
    selected_line.to_excel(writer, sheet_name='selected_line', index=False)
    scored_variants.to_excel(writer, sheet_name='all_scores', index=False)
    variants.to_excel(writer, sheet_name='variants_40', index=False)

with pd.ExcelWriter(squad_path) as writer:
    symbolic_squad.to_excel(writer, sheet_name='symbolic_squad', index=False)
    ranked_players.to_excel(writer, sheet_name='all_candidates', index=False)

line_path, squad_path